# 08 — Tanore & Manda Final Point-Level Validation

এই notebook **03 Classification আবার run করবে না**।

এটি existing 03 outputs ব্যবহার করে final validationকে original reference-point level-এ নিয়ে যাবে।

## কেন দরকার
03 Classification-এর validation CSV-তে একটি original reference point-এর 6 m neighborhood থেকে একাধিক 3 m pixel row আছে।  
`sample_id` = extracted pixel row  
`source_feature` = original reference point

Final independent validation therefore হবে:

- Tanore = **97 reference points**
- Manda = **96 reference points**

## কী করবে
- Existing `Q1_Validation_Predictions.csv` read করবে
- Stream-specific validation sample table থেকে `source_feature` attach করবে
- Pixel probabilities `source_feature` অনুযায়ী average করবে
- RF/XGBoost-এর original decision threshold ব্যবহার করবে
- RuleBased-এর জন্য within-point majority hard prediction ব্যবহার করবে
- Point-level OA, Precision, Recall, F1, MCC, ROC-AUC, PR-AUC, Brier, Kappa
- Point-bootstrap 95% CI
- Point-level confusion matrices
- Fused-Hybrid vs Planet-only exact McNemar tests
- Benjamini-Hochberg FDR adjusted p-values
- CSV + Excel + 300-DPI figures save করবে

**No map retraining or reclassification is required.**

### GitHub execution note
This notebook preserves the publication analysis logic. Local absolute paths were replaced with the portable `BORO_PROJECT_ROOT` setting. Run Jupyter from the repository root or set that environment variable before execution. Generated figures and tables are written below `Outputs/`; licensed source imagery is not included.


In [ ]:
# CELL 1 — Imports, project paths, and settings

from pathlib import Path
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from scipy.stats import binomtest
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    cohen_kappa_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    os.environ.get(
        "BORO_PROJECT_ROOT",
        str(Path.cwd()),
    )
)

AREAS = ["Tanore", "Manda"]
STREAMS = ["FusedHybrid", "PlanetOnly"]
MODELS = ["RuleBased", "RandomForest", "XGBoost"]

REFERENCE_ID_COL = "source_feature"
BOOTSTRAP_REPLICATES = 2000
RANDOM_SEED = 42

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "Outputs"
    / "FINAL_PUBLICATION"
    / "PointLevel_Validation"
)

TABLE_DIR = OUTPUT_ROOT / "tables"
FIGURE_DIR = OUTPUT_ROOT / "figures"

for folder in [OUTPUT_ROOT, TABLE_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root :", PROJECT_ROOT)
print("Output folder:", OUTPUT_ROOT)

In [ ]:
# CELL 2 — Find existing Classification_Q1 folders robustly

def find_classification_root(area):
    candidates = [
        PROJECT_ROOT / "Outputs" / area / "Classification_Q1",
        PROJECT_ROOT / "classification_q1" / area,
        PROJECT_ROOT / "Outputs" / "Classification_Q1" / area,
    ]

    for candidate in candidates:
        if (
            (candidate / "tables").exists()
            and (candidate / "models").exists()
        ):
            return candidate

    # Allow tables-only fallback.
    for candidate in candidates:
        if (candidate / "tables").exists():
            return candidate

    raise FileNotFoundError(
        f"Could not locate Classification_Q1 output for {area}.\n"
        f"Checked:\n" + "\n".join(str(x) for x in candidates)
    )

CLASS_ROOTS = {
    area: find_classification_root(area)
    for area in AREAS
}

for area, root in CLASS_ROOTS.items():
    print(area, "→", root)

In [ ]:
# CELL 3 — Load saved pixel-level validation predictions and attach source_feature

def required_file(path, label):
    if not path.exists():
        raise FileNotFoundError(
            f"Required {label} file not found:\n{path}"
        )
    return path


def load_area_inputs(area):
    root = CLASS_ROOTS[area]
    table_dir = root / "tables"

    pred_path = required_file(
        table_dir / "Q1_Validation_Predictions.csv",
        "validation prediction",
    )

    pixel_predictions = pd.read_csv(pred_path)

    required_pred_cols = {
        "sample_id",
        "true_class",
        "stream",
        "model",
        "probability",
        "prediction",
    }

    missing = required_pred_cols - set(pixel_predictions.columns)

    if missing:
        raise ValueError(
            f"{pred_path.name} missing columns: {sorted(missing)}"
        )

    stream_tables = {}

    for stream in STREAMS:
        sample_path = required_file(
            table_dir / f"Q1_{stream}_Validation_Samples.csv",
            f"{stream} validation sample",
        )

        samples = pd.read_csv(sample_path)

        required_sample_cols = {
            "sample_id",
            REFERENCE_ID_COL,
            "class",
            "group",
        }

        missing_sample = (
            required_sample_cols
            - set(samples.columns)
        )

        if missing_sample:
            raise ValueError(
                f"{sample_path.name} missing columns: {sorted(missing_sample)}"
            )

        # one pixel sample_id = one row
        samples = (
            samples
            .drop_duplicates("sample_id")
            .copy()
        )

        stream_tables[stream] = samples

    enriched_parts = []

    for stream in STREAMS:
        pred_sub = pixel_predictions[
            pixel_predictions["stream"] == stream
        ].copy()

        samples = stream_tables[stream]

        keep_cols = [
            "sample_id",
            REFERENCE_ID_COL,
            "class",
            "group",
        ]

        for optional_col in ["x", "y"]:
            if optional_col in samples.columns:
                keep_cols.append(optional_col)

        merged = pred_sub.merge(
            samples[keep_cols],
            on="sample_id",
            how="left",
            validate="many_to_one",
            suffixes=("", "_sample"),
        )

        if merged[REFERENCE_ID_COL].isna().any():
            n_missing = int(
                merged[REFERENCE_ID_COL]
                .isna()
                .sum()
            )
            raise ValueError(
                f"{area} {stream}: {n_missing} prediction rows could not "
                f"be linked to source_feature."
            )

        # Verify saved true label agrees with sample table.
        if not np.array_equal(
            merged["true_class"].astype(int).to_numpy(),
            merged["class"].astype(int).to_numpy(),
        ):
            raise ValueError(
                f"{area} {stream}: true_class does not match validation sample class."
            )

        enriched_parts.append(merged)

    enriched = pd.concat(
        enriched_parts,
        ignore_index=True,
    )

    return enriched, stream_tables


area_inputs = {}

summary_rows = []

for area in AREAS:
    enriched, stream_tables = load_area_inputs(area)
    area_inputs[area] = enriched

    for stream in STREAMS:
        sample_table = stream_tables[stream]

        summary_rows.append({
            "area": area,
            "stream": stream,
            "pixel_rows": len(sample_table),
            "reference_points": int(
                sample_table[REFERENCE_ID_COL].nunique()
            ),
            "rice_points": int(
                sample_table[
                    [REFERENCE_ID_COL, "class"]
                ]
                .drop_duplicates(REFERENCE_ID_COL)["class"]
                .eq(1)
                .sum()
            ),
            "nonrice_points": int(
                sample_table[
                    [REFERENCE_ID_COL, "class"]
                ]
                .drop_duplicates(REFERENCE_ID_COL)["class"]
                .eq(0)
                .sum()
            ),
        })

input_summary = pd.DataFrame(summary_rows)
display(input_summary)

expected = {
    "Tanore": 97,
    "Manda": 96,
}

for area, expected_n in expected.items():
    observed = int(
        input_summary[
            input_summary["area"] == area
        ]["reference_points"]
        .iloc[0]
    )

    if observed != expected_n:
        raise ValueError(
            f"{area}: expected {expected_n} reference points, found {observed}."
        )

print("\n✅ Original validation points recovered correctly.")

In [ ]:
# CELL 4 — Recover original RF/XGB decision thresholds

def infer_binary_threshold(probability, prediction):
    probability = np.asarray(probability, dtype=float)
    prediction = np.asarray(prediction, dtype=int)

    p0 = probability[prediction == 0]
    p1 = probability[prediction == 1]

    if len(p0) == 0 or len(p1) == 0:
        return 0.5

    upper_zero = float(np.max(p0))
    lower_one = float(np.min(p1))

    if upper_zero <= lower_one:
        return float((upper_zero + lower_one) / 2.0)

    # Fallback: find threshold minimizing mismatch.
    candidates = np.unique(probability)

    best_threshold = 0.5
    best_error = np.inf

    for threshold in candidates:
        pred = (probability >= threshold).astype(int)
        error = np.mean(pred != prediction)

        if error < best_error:
            best_error = error
            best_threshold = float(threshold)

    return best_threshold


threshold_rows = []

for area in AREAS:
    data = area_inputs[area]

    for stream in STREAMS:
        for model in ["RandomForest", "XGBoost"]:

            sub = data[
                (data["stream"] == stream)
                & (data["model"] == model)
            ]

            threshold = infer_binary_threshold(
                sub["probability"],
                sub["prediction"],
            )

            reproduced = (
                sub["probability"].to_numpy(dtype=float)
                >= threshold
            ).astype(int)

            agreement = float(
                np.mean(
                    reproduced
                    == sub["prediction"].to_numpy(dtype=int)
                )
            )

            threshold_rows.append({
                "area": area,
                "stream": stream,
                "model": model,
                "decision_threshold_recovered": threshold,
                "pixel_prediction_reproduction_rate": agreement,
            })

threshold_table = pd.DataFrame(threshold_rows)

display(threshold_table.round(6))

if (
    threshold_table[
        "pixel_prediction_reproduction_rate"
    ] < 0.999
).any():
    print(
        "⚠ Some thresholds did not reproduce 99.9% of saved hard predictions. "
        "The notebook will still use the recovered threshold, but review that row."
    )
else:
    print("\n✅ RF/XGB decision thresholds recovered from saved predictions.")

In [ ]:
# CELL 5 — Aggregate to ONE prediction per original reference point

def get_threshold(area, stream, model):
    return float(
        threshold_table[
            (threshold_table["area"] == area)
            & (threshold_table["stream"] == stream)
            & (threshold_table["model"] == model)
        ]["decision_threshold_recovered"]
        .iloc[0]
    )


point_frames = []

for area in AREAS:
    data = area_inputs[area]

    for stream in STREAMS:
        for model in MODELS:

            sub = data[
                (data["stream"] == stream)
                & (data["model"] == model)
            ].copy()

            agg_spec = {
                "true_class": ("true_class", "first"),
                "group": ("group", "first"),
                "probability": ("probability", "mean"),
                "pixel_prediction_mean": ("prediction", "mean"),
                "n_pixel_rows": ("prediction", "size"),
            }

            if "x" in sub.columns:
                agg_spec["x"] = ("x", "mean")

            if "y" in sub.columns:
                agg_spec["y"] = ("y", "mean")

            point = (
                sub
                .groupby(
                    REFERENCE_ID_COL,
                    as_index=False,
                )
                .agg(**agg_spec)
            )

            if model in ["RandomForest", "XGBoost"]:
                threshold = get_threshold(
                    area,
                    stream,
                    model,
                )

                point["decision_threshold"] = threshold

                point["prediction"] = (
                    point["probability"]
                    >= threshold
                ).astype("uint8")

                prediction_method = (
                    "mean probability >= original model threshold"
                )

            else:
                # RuleBased hard rule is not equivalent to thresholding its
                # smooth probability score; use majority hard-pixel decision.
                point["decision_threshold"] = np.nan

                point["prediction"] = (
                    point["pixel_prediction_mean"]
                    >= 0.5
                ).astype("uint8")

                prediction_method = (
                    "majority hard-pixel rule within reference point"
                )

            point["area"] = area
            point["stream"] = stream
            point["model"] = model
            point["prediction_method"] = prediction_method

            point_frames.append(point)

point_predictions = pd.concat(
    point_frames,
    ignore_index=True,
)

count_check = (
    point_predictions
    .groupby(
        ["area", "stream", "model"]
    )[REFERENCE_ID_COL]
    .nunique()
    .reset_index(name="n_reference_points")
)

display(count_check)

print("\n✅ Final point-level predictions created.")

In [ ]:
# CELL 6 — Metrics and point-bootstrap 95% confidence intervals

def metric_row(y_true, probability, prediction):
    tn, fp, fn, tp = confusion_matrix(
        y_true,
        prediction,
        labels=[0, 1],
    ).ravel()

    specificity = (
        tn / (tn + fp)
        if (tn + fp)
        else np.nan
    )

    return {
        "OA": accuracy_score(y_true, prediction),
        "Balanced_Accuracy": balanced_accuracy_score(
            y_true,
            prediction,
        ),
        "Precision": precision_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "Recall": recall_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "Specificity": specificity,
        "F1": f1_score(
            y_true,
            prediction,
            zero_division=0,
        ),
        "Kappa": cohen_kappa_score(
            y_true,
            prediction,
        ),
        "MCC": matthews_corrcoef(
            y_true,
            prediction,
        ),
        "ROC_AUC": roc_auc_score(
            y_true,
            probability,
        ),
        "PR_AUC": average_precision_score(
            y_true,
            probability,
        ),
        "Brier": brier_score_loss(
            y_true,
            probability,
        ),
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
    }


def bootstrap_f1_ci(frame, replicates, seed):
    rng = np.random.default_rng(seed)
    n = len(frame)
    values = []

    for _ in range(replicates):
        idx = rng.integers(
            0,
            n,
            size=n,
        )

        sampled = frame.iloc[idx]

        values.append(
            f1_score(
                sampled["true_class"],
                sampled["prediction"],
                zero_division=0,
            )
        )

    return tuple(
        np.quantile(
            values,
            [0.025, 0.975],
        )
    )


metric_rows = []

for (
    area,
    stream,
    model,
), sub in point_predictions.groupby(
    ["area", "stream", "model"]
):

    y_true = sub["true_class"].to_numpy(dtype=int)
    prob = sub["probability"].to_numpy(dtype=float)
    pred = sub["prediction"].to_numpy(dtype=int)

    metrics = metric_row(
        y_true,
        prob,
        pred,
    )

    ci_low, ci_high = bootstrap_f1_ci(
        sub,
        BOOTSTRAP_REPLICATES,
        RANDOM_SEED,
    )

    metric_rows.append({
        "area": area,
        "stream": stream,
        "model": model,
        "n_reference_points": len(sub),
        "F1_CI_low": ci_low,
        "F1_CI_high": ci_high,
        **metrics,
    })

point_metrics = pd.DataFrame(
    metric_rows
)

display(
    point_metrics[
        [
            "area",
            "stream",
            "model",
            "n_reference_points",
            "OA",
            "Precision",
            "Recall",
            "F1",
            "F1_CI_low",
            "F1_CI_high",
            "MCC",
            "ROC_AUC",
            "PR_AUC",
            "Brier",
        ]
    ].round(4)
)

print("\n✅ Final point-level validation metrics complete.")

In [ ]:
# CELL 7 — Point-level Fused-Hybrid vs Planet-only exact McNemar tests

def benjamini_hochberg(p_values):
    p = np.asarray(p_values, dtype=float)
    n = len(p)

    order = np.argsort(p)
    ranked = p[order]

    adjusted_ranked = (
        ranked
        * n
        / np.arange(1, n + 1)
    )

    adjusted_ranked = np.minimum.accumulate(
        adjusted_ranked[::-1]
    )[::-1]

    adjusted_ranked = np.clip(
        adjusted_ranked,
        0,
        1,
    )

    adjusted = np.empty(n, dtype=float)
    adjusted[order] = adjusted_ranked

    return adjusted


comparison_rows = []

for area in AREAS:
    for model in MODELS:

        fused = point_predictions[
            (point_predictions["area"] == area)
            & (point_predictions["stream"] == "FusedHybrid")
            & (point_predictions["model"] == model)
        ][
            [
                REFERENCE_ID_COL,
                "true_class",
                "prediction",
            ]
        ].rename(
            columns={
                "prediction": "prediction_fused"
            }
        )

        planet = point_predictions[
            (point_predictions["area"] == area)
            & (point_predictions["stream"] == "PlanetOnly")
            & (point_predictions["model"] == model)
        ][
            [
                REFERENCE_ID_COL,
                "prediction",
            ]
        ].rename(
            columns={
                "prediction": "prediction_planet"
            }
        )

        merged = fused.merge(
            planet,
            on=REFERENCE_ID_COL,
            how="inner",
        )

        fused_correct = (
            merged["prediction_fused"]
            == merged["true_class"]
        )

        planet_correct = (
            merged["prediction_planet"]
            == merged["true_class"]
        )

        fused_only = int(
            (fused_correct & ~planet_correct).sum()
        )

        planet_only = int(
            (~fused_correct & planet_correct).sum()
        )

        discordant = fused_only + planet_only

        p_value = (
            binomtest(
                fused_only,
                discordant,
                p=0.5,
                alternative="two-sided",
            ).pvalue
            if discordant > 0
            else 1.0
        )

        fused_f1 = f1_score(
            merged["true_class"],
            merged["prediction_fused"],
            zero_division=0,
        )

        planet_f1 = f1_score(
            merged["true_class"],
            merged["prediction_planet"],
            zero_division=0,
        )

        comparison_rows.append({
            "area": area,
            "model": model,
            "n_paired_reference_points": len(merged),
            "fused_only_correct": fused_only,
            "planet_only_correct": planet_only,
            "discordant": discordant,
            "mcnemar_exact_p": p_value,
            "fused_F1": fused_f1,
            "planet_F1": planet_f1,
            "delta_F1_fused_minus_planet": (
                fused_f1 - planet_f1
            ),
        })

stream_comparison = pd.DataFrame(
    comparison_rows
)

stream_comparison[
    "mcnemar_FDR_BH_p"
] = benjamini_hochberg(
    stream_comparison[
        "mcnemar_exact_p"
    ].to_numpy()
)

stream_comparison[
    "significant_FDR_0.05"
] = (
    stream_comparison[
        "mcnemar_FDR_BH_p"
    ] < 0.05
)

display(
    stream_comparison.round(4)
)

print("\n✅ Point-level paired stream comparisons complete.")

In [ ]:
# CELL 8 — Publication-ready confusion matrices

for row in point_metrics.itertuples(index=False):

    sub = point_predictions[
        (point_predictions["area"] == row.area)
        & (point_predictions["stream"] == row.stream)
        & (point_predictions["model"] == row.model)
    ]

    cm = confusion_matrix(
        sub["true_class"],
        sub["prediction"],
        labels=[0, 1],
    )

    fig, ax = plt.subplots(
        figsize=(5.2, 4.6)
    )

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=[
            "Non-Boro",
            "Boro",
        ],
    )

    disp.plot(
        ax=ax,
        colorbar=False,
    )

    ax.set_title(
        f"{row.area} | {row.stream} | {row.model}\n"
        f"Reference-point validation (n={row.n_reference_points})"
    )

    fig.tight_layout()

    fig.savefig(
        FIGURE_DIR
        / (
            f"PointLevel_CM_{row.area}_"
            f"{row.stream}_{row.model}.png"
        ),
        dpi=300,
        bbox_inches="tight",
    )

    plt.close(fig)

print("✅ Point-level confusion matrices saved.")

In [ ]:
# CELL 9 — Main article figure: F1 by area, stream, and model

plot_df = point_metrics.copy()

plot_df["label"] = (
    plot_df["area"]
    + " | "
    + plot_df["stream"]
    + " | "
    + plot_df["model"]
)

plot_df = plot_df.sort_values(
    ["area", "F1"]
)

fig, ax = plt.subplots(
    figsize=(10.5, 7.2)
)

ax.barh(
    plot_df["label"],
    plot_df["F1"],
)

ax.set_xlabel(
    "Reference-point F1 score"
)

ax.set_title(
    "Final Within-Area Point-Level Validation"
)

minimum = float(
    plot_df["F1"].min()
)

ax.set_xlim(
    max(0, minimum - 0.08),
    1.01,
)

fig.tight_layout()

fig.savefig(
    FIGURE_DIR
    / "PointLevel_WithinArea_F1_Comparison.png",
    dpi=300,
    bbox_inches="tight",
)

plt.show()

In [ ]:
# CELL 10 — Save final publication tables

input_summary.to_csv(
    TABLE_DIR
    / "PointLevel_Input_Summary.csv",
    index=False,
)

threshold_table.to_csv(
    TABLE_DIR
    / "Recovered_Decision_Thresholds.csv",
    index=False,
)

point_predictions.to_csv(
    TABLE_DIR
    / "PointLevel_Validation_Predictions.csv",
    index=False,
)

point_metrics.to_csv(
    TABLE_DIR
    / "PointLevel_Validation_Metrics.csv",
    index=False,
)

stream_comparison.to_csv(
    TABLE_DIR
    / "PointLevel_Fused_vs_Planet_McNemar.csv",
    index=False,
)

excel_path = (
    OUTPUT_ROOT
    / "Final_PointLevel_Validation_Results.xlsx"
)

with pd.ExcelWriter(
    excel_path,
    engine="openpyxl",
) as writer:

    input_summary.to_excel(
        writer,
        sheet_name="Input_Summary",
        index=False,
    )

    threshold_table.to_excel(
        writer,
        sheet_name="Thresholds",
        index=False,
    )

    point_metrics.to_excel(
        writer,
        sheet_name="Point_Metrics",
        index=False,
    )

    stream_comparison.to_excel(
        writer,
        sheet_name="Fused_vs_Planet",
        index=False,
    )

    point_predictions.to_excel(
        writer,
        sheet_name="Point_Predictions",
        index=False,
    )

print("\n" + "=" * 78)
print("✅ FINAL POINT-LEVEL VALIDATION COMPLETE")
print("=" * 78)
print("Excel  :", excel_path)
print("Tables :", TABLE_DIR)
print("Figures:", FIGURE_DIR)

# Final article use

Main paper-এ এই notebook থেকে ব্যবহার করবেন:

1. `PointLevel_Validation_Metrics.csv`
2. `PointLevel_Fused_vs_Planet_McNemar.csv`
3. `PointLevel_WithinArea_F1_Comparison.png`
4. Selected point-level confusion matrices

Pixel-level 03 accuracy table main manuscript-এ final independent accuracy হিসেবে ব্যবহার করবেন না।

Maps, mapped area, SHAP/feature-importance এবং image-fusion outputs 03/02 notebooks থেকে আগের মতোই ব্যবহার করা যাবে।